In [1]:
#importing libraries 
import pandas as pd
import numpy as np
import os 

os.makedirs('~/Desktop', exist_ok=True)
print('ready')

ready


In [5]:
#loading the data from github 
songs_url = 'https://raw.githubusercontent.com/Liviah27/Empirical-Project-Data-Science-26/refs/heads/raw/audio_features2.csv'
df_songs = pd.read_csv(songs_url)

macro_url = 'https://raw.githubusercontent.com/Liviah27/Empirical-Project-Data-Science-26/refs/heads/raw/economic_fred.csv'
df_economic = pd.read_csv(macro_url)

print(f'songs: {len(df_songs):,} rows')
print(f'economic: {len(df_economic)} years')
print('\nsong columns:', df_songs.columns.tolist())
print('economic columns:', df_economic.columns.tolist())

songs: 5,100 rows
economic: 51 years

song columns: ['year', 'position', 'title', 'artist', 'bpm', 'valence']
economic columns: ['year', 'gdp_growth_pct', 'unemployment_rate', 'recession']


In [7]:
#cleaning the audio features by removing any values outside the valid range (BPM is almost always between 60-220 BPM)
#valence is defined by spotify as 0-1

df_songs.loc[(df_songs['bpm']< 60) | (df_songs['bpm'] > 220), 'bpm'] = np.nan
df_songs.loc[(df_songs['valence'] < 0) | (df_songs['valence'] > 1), 'valence'] = np.nan

print(f'songs with valid BPM: {df_songs['bpm'].notna().sum():,} ({df_songs['bpm'].notna().mean():.1%})')
print(f'songs with valid valence: {df_songs['valence'].notna().sum():,} ({df_songs['valence'].notna().mean():.1%})')

df_songs.to_csv('~/Desktop/songs_merged.csv', index=False)
print('\nsaved: songs_merged.csv')


songs with valid BPM: 705 (13.8%)
songs with valid valence: 705 (13.8%)

saved: songs_merged.csv


In [15]:
#aggregate to annual level, computing one BPM and valence value per year by averaging 
#use position weighted averaging, chart position 1 getsweight 100 and position 100 gets weight 1
#meaning bigger hits count more as it helps reflect how much people were actually listening to it

df_songs['weight'] = 101 - df_songs['position']

def weighted_mean(vals, weights):
    mask = vals.notna() & weights.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(vals[mask], weights=weights[mask])

records = []
for year, grp in df_songs.groupby('year'):
    records.append({
        'year': year,
        'mean_bpm': grp['bpm'].mean(),
        'weighted_mean_bpm': weighted_mean(grp['bpm'], grp['weight']),
        'top10_mean_bpm': grp.nsmallest(10, 'position')['bpm'].mean(),
        'mean_valence': grp['valence'].mean(),
        'weighted_mean_valence': weighted_mean(grp['valence'], grp['weight']),
        'top10_mean_valence': grp.nsmallest(10, 'position')['valence'].mean(),
        'n_songs_with_data': grp['bpm'].notna().sum(),
    })

df_annual = pd.DataFrame(records)
print(f'annual data: {len(df_annual)} years')
print(df_annual.head())

annual data: 51 years
   year    mean_bpm  weighted_mean_bpm  top10_mean_bpm  mean_valence  \
0  1975  113.931462         115.975746         101.176      0.759846   
1  1976  115.660222         113.974555         104.413      0.732667   
2  1977  117.080647         119.013764         109.425      0.602118   
3  1978  113.395778         112.809014         101.785      0.602222   
4  1979  119.525611         120.192649         124.031      0.474222   

   weighted_mean_valence  top10_mean_valence  n_songs_with_data  
0               0.766815            0.797667                 13  
1               0.700095            0.964000                  9  
2               0.661878            0.795500                 17  
3               0.602005            0.942000                  9  
4               0.484898            0.631667                 18  


In [25]:
#merging with macroeconomic data
df_final = df_annual.merge(df_economic, on='year', how='inner')

#as music is recorded months before release we will test last years macroeconomics against this years music
df_final = df_final.sort_values('year')

df_final['unemployment_lag1'] = df_final['unemployment_rate'].shift(1)
df_final['gdp_growth_lag1'] = df_final['gdp_growth_pct'].shift(1)

print(f'Final dataset: {len(df_final)} years ({df_final["year"].min()}-{df_final["year"].max()})')
print('\nFull annual dataset:')
print(df_final[['year', 'mean_bpm', 'mean_valence', 'unemployment_rate', 'gdp_growth_pct', 'recession']].to_string(index=False))

df_final.to_csv('~/Desktop/annual_merged.csv', index=False)
print('\nsaved: annual_merged.csv')


Final dataset: 51 years (1975-2025)

Full annual dataset:
 year   mean_bpm  mean_valence  unemployment_rate  gdp_growth_pct  recession
 1975 113.931462      0.759846           8.475000        2.554795          1
 1976 115.660222      0.732667           7.700000        4.311516          0
 1977 117.080647      0.602118           7.050000        5.013271          0
 1978 113.395778      0.602222           6.066667        6.659560          0
 1979 119.525611      0.474222           5.850000        1.284088          0
 1980 116.717062      0.626437           7.175000       -0.039052          1
 1981 121.505429      0.535357           7.616667        1.299825          1
 1982 113.833400      0.560867           9.708333       -1.443184          1
 1983 119.322467      0.533553           9.600000        7.899664          0
 1984 119.466000      0.376255           7.508333        5.575644          0
 1985 116.885750      0.585375           7.191667        4.182460          0
 1986 108.196000  